# Exercise 2.2.1.4 — fill in the agent class

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `2.2.1 DQN`  
**Notebook:** `2.2.1_Deep_Q_Networks_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=2.2.1.4](https://delta-drills.vercel.app/?arena_exercise=2.2.1.4)


# [2.2.1] - Deep Q Networks (exercises)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/20_[2.2.1]_Deep_Q_Networks)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part21_dqn/2.2.1_Deep_Q_Networks_exercises.ipynb?t=20250917) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part21_dqn/2.2.1_Deep_Q_Networks_solutions.ipynb?t=20250917)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-22.png" width="350">

# Introduction

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf). 

<!-- 
You'll also implement Vanilla Policy Gradient (VPG), the first policy gradient algorithm upon which many modern RL algorithms are based (including PPO).
This was the first working version of deep learning in the RL setting.

At the time, the idea that convolutional neural networks could look at Atari game pixels and "see" gameplay-relevant features like a Space Invader was new and noteworthy. In 2022, we take for granted that convnets work, so we're going to focus on the RL aspect and not the vision aspect today. -->

## Content & Learning Objectives


### 1️⃣ DQN

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper Playing Atari with [Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf).

You'll apply the technique of DQN to master the famous CartPole environment (below), and then (if you have time) move on to harder challenges like Acrobot and MountainCar.

> ##### Learning Objectives
>
> - Understand the DQN algorithm
> - Learn more about RL debugging, and build probe environments to debug your agents
> - Create a replay buffer to store environment transitions
> - Implement DQN using PyTorch, on the CartPole environment

## Setup (don't read, just run!)

In [ ]:
from __future__ import annotations

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter2_rl"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import jaxtyping
except:
    %pip install wandb==0.18.7 einops "gymnasium[atari, accept-rom-license, other]==0.29.0" pygame jaxtyping

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import TypeAlias

import gymnasium as gym
import numpy as np
import torch as t
import wandb
from gymnasium.spaces import Box, Discrete
from jaxtyping import Bool, Float, Int
from torch import Tensor, nn
from tqdm import tqdm, trange

warnings.filterwarnings("ignore")

Arr = np.ndarray

# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part21_dqn"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part21_dqn.tests as tests
import part21_dqn.utils as utils
from part1_intro_to_rl.solutions import Environment, Norvig, Toy, find_optimal_policy
from part1_intro_to_rl.utils import set_global_seeds
from part21_dqn.utils import make_env
from plotly_utils import cliffwalk_imshow, line, plot_cartpole_obs_and_dones
from rl_utils import generate_and_plot_trajectory

device = t.device(
    "mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu"
)

# 1️⃣ Deep Q-Learning

> ##### Learning Objectives
>
> - Understand the DQN algorithm
> - Learn more about RL debugging, and build probe environments to debug your agents
> - Create a replay buffer to store environment transitions
> - Implement DQN using PyTorch, on the CartPole environment

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf).

At the time, the paper was very exciting: The agent would play the game by only looking at the same screen pixel data that a human player would be looking at, rather than a description of where the enemies in the game world are. The idea that convolutional neural networks could look at Atari game pixels and "see" gameplay-relevant features like a Space Invader was new and noteworthy. In 2022, we take for granted that convnets work, so we're going to focus on the RL aspect solely, and not the vision component.

## Optional Readings

* [Deep Q Networks Explained](https://www.lesswrong.com/posts/kyvCNgx9oAwJCuevo/deep-q-networks-explained) (25 minutes)
    * A high-level distillation as to how DQN works.
    * Read sections 1-4 (further sections optional).
* [Andy Jones - Debugging RL, Without the Agonizing Pain](https://andyljones.com/posts/rl-debugging.html) (10 minutes)
    * Useful tips for debugging your code when it's not working.
    * Read up to (not including) the Common Fixes section. Also read the Practical Advice section, up to and including "Use probe agents". The rest of the post is optional, and you're recommended to come back to it near the end if you're stuck.
    * The "probe environments" (a collection of simple environments of increasing complexity) section will be our first line of defense against bugs, you'll implement these in exercises below.

### Interesting Resources (not required reading)

- [An Outsider's Tour of Reinforcement Learning](http://www.argmin.net/2018/06/25/outsider-rl/) - comparison of RL techniques with the engineering discipline of control theory.
- [Towards Characterizing Divergence in Deep Q-Learning](https://arxiv.org/pdf/1903.08894.pdf) - analysis of what causes learning to diverge
- [Divergence in Deep Q-Learning: Tips and Tricks](https://amanhussain.com/post/divergence-deep-q-learning/) - includes some plots of average returns for comparison
- [Deep RL Bootcamp](https://sites.google.com/view/deep-rl-bootcamp/lectures) - 2017 bootcamp with video and slides. Good if you like videos.
- [DQN debugging using OpenAI gym Cartpole](https://adgefficiency.com/dqn-debugging/) - random dude's adventures in trying to get it to work.
- [CleanRL DQN](https://github.com/vwxyzjn/cleanrl) - single file implementations of RL algorithms. Your starter code today is based on this; try not to spoiler yourself by looking at the solutions too early!
- [Deep Reinforcement Learning Doesn't Work Yet](https://www.alexirpan.com/2018/02/14/rl-hard.html) - 2018 article describing difficulties preventing industrial adoption of RL.
- [Deep Reinforcement Learning Works - Now What?](https://tesslerc.github.io/posts/drl_works_now_what/) - 2020 response to the previous article highlighting recent progress.
- [Seed RL](https://github.com/google-research/seed_rl) - example of distributed RL using Docker and GCP.

### Conceptual overview of DQN

DQN is the natural extension of Q-Learning into the domain of deep learning. The main difference is that, instead of a table to store all the Q-values for each state-action pair, we train a neural network to learn this function for us. The usual implementation (which we'll use here) is for the Q-network to take the state as input, and output a vector of optimalQ-values for each action, i.e. we're learning the function:

$$
s \to (Q^*(s, a_1), ..., Q^*(s, a_n))
$$

Below is an algorithm showing the conceptual overview of DQN. We cycle through the following process:

* Generate a batch of experiences using our current policy, by **epsilon-greedy sampling** (i.e. we mostly take the action with the highest Q-value, but occasionally take a random action to encourage exploration). Store these experiences in the **replay buffer**.
* Use these values to calculate a **TD (temporal difference) error**, and update our network.
    * To increase stability, we also have a **target network** we use for the "next step" part of the TD error. This is a lagged copy of the Q-network (i.e. we update our Q-network via gradient descent, and then every so often we copy the Q-network weights over to our target network).
* Repeat this until convergence.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ppo-alg-conceptual-2.png" width="750">

### Fast Feedback Loops

We want to have faster feedback loops, and learning from Atari pixels doesn't achieve that. It might take 15 minutes per training run to get an agent to do well on Breakout, and that's if your implementation is relatively optimized. Even waiting 5 minutes to learn Pong from pixels is going to limit your ability to iterate, compared to using environments that are as simple as possible.

### CartPole

The classic environment "CartPole-v1" is simple to understand, yet hard enough for a RL agent to be interesting, by the end of the day your agent will be able to do this and more! (Click to watch!)


[![CartPole](https://img.youtube.com/vi/46wjA6dqxOM/0.jpg)](https://www.youtube.com/watch?v=46wjA6dqxOM "CartPole")

If you'd like to try the CartPole environment yourself, [click here to open the simulation](https://davidquarel.github.io/cartpole) in a new tab. 
* Use Left/Right arrow keys to move the cart, 
* R to reset, 
* Q to quit. 
* Use F/S to make the simulation faster/slower. 

Unlike the real CartPole environment, this simulation will not terminate the episode if the pole falls over.
We've also cheated here and added a hidden third no-op action, such that if no button is pressed, no force is applied to the cart.
This makes the simulation a bit easier for you as the human to play.
The real cartpole environment doesn't act like this: the agent must choose to push the cart either left or right on each timestep.

The description of the task is [here](https://gymnasium.farama.org/environments/classic_control/cart_pole/). Note that unlike the previous environments, the observation here is now continuous. You can see the source for CartPole [here](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.0/gymnasium/envs/classic_control/cartpole.py); don't worry about the implementation but do read the documentation to understand the format of the actions and observations.

The simple physics involved would be very easy for a model-based algorithm to fit, (this is a common assignment in control theory using [proportional-integral-derivative](https://en.wikipedia.org/wiki/PID_controller) (PID) controllers) but today we're doing it model-free: your agent has no idea that these observations represent positions or velocities, and it has no idea what the laws of physics are. The network has to learn in which direction to bump the cart in response to the current state of the world.

Each environment can have different versions registered to it. By consulting [the Gym source](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.0/gymnasium/envs/__init__.py) you can see that CartPole-v0 and CartPole-v1 are the same environment, except that v1 has longer episodes. Again, a minor change like this can affect what algorithms score well; an agent might consistently survive for 200 steps in an unstable fashion that means it would fall over if ran for 500 steps.

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

print(env.action_space)  # 2 actions: left and right
print(env.observation_space)  # Box(4): each action can take a continuous range of values

### Outline of the Exercises

The exercises are roughly split into 4 sections:

1. Implement the Q-network that maps a state to an estimated value for each action.
2. Implement a replay buffer to store experiences $e_t = (s_t, a_t, r_{t+1}, d_{t+1}, s_{t+1})$.
3. Implement the policy which chooses actions based on the Q-network, plus epsilon greedy randomness to encourage exploration.
4. Piece everything together into a training loop and train your agent.

## The Q-Network

The Q-Network takes in an observation $s$ and outputs a vector $[Q^*(s, a^1), \ldots Q^*(s,a^n)]$ representing an estimate of the optimal Q-value for the given state $s$, and each possible action $\mathcal{A} = \{a^1, \ldots, a^n\}$. This replaces our Q-value table used in Q-learning.

For best results, the architecture of the Q-network can be customized to each particular problem. For example, [the architecture of OpenAI Five](https://cdn.openai.com/research-covers/openai-five/network-architecture.pdf) used to play DOTA 2 is pretty complex and involves LSTMs.

For learning from pixels, a simple convolutional network and some fully connected layers does quite well. Where we have already processed features here, it's even easier: an MLP of this size should be plenty large for any environment today.

Implement the Q-network using a standard MLP, constructed of alternating Linear and ReLU layers.
The size of the input will match the dimensionality of the observation space, and the size of the output will match the number of actions to choose from (associating a reward to each.)
The dimensions of the hidden_sizes are provided.

Here is a diagram of what our particular Q-Network will look like for CartPole (you can open it in a new tab if it's hard to see clearly):

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/mermaid-diagram-2024-11-28-200952.svg" style="background-color: #fffbe0; padding: 10px" width="160"><br>

<!-- 
flowchart TD
    A["Input (num_obs,)"] -> B["Linear(num_obs, 120)"]
    B -> C[ReLU]
    C -> D["Linear(120, 84)"]
    D -> E[ReLU]
    E -> F["Linear(84, num_actions)"]
    F -> G["Output (num_actions,)"]
{
  "theme": "default",
  "themeVariables": {
    "fontSize": "22px"
  }
}
-->

<details>
<summary>Question - why do we not include a ReLU at the end?</summary>

If you end with a ReLU, then your network can only predict 0 or positive Q-values. This will cause problems as soon as you encounter an environment with negative rewards, or you try to do some scaling of the rewards.

</details>

<details>
<summary>Question - since CartPole-v1 gives +1 reward on every timestep, why do you think the network doesn't just learn the constant +1 function regardless of observation?</summary>

The network is learning Q-values (the sum of all future expected discounted rewards from this state/action pair), not rewards. Correspondingly, once the agent has learned a good policy, the Q-value associated with state action pair (pole is slightly left of vertical, move cart left) should be large, as we would expect a long episode (and correspondingly lots of reward) by taking actions to help to balance the pole. Pairs like (cart near right boundary, move cart right) cause the episode to terminate, and as such the network will learn low Q-values.

</details>

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "2.2.1.4"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part21_dqn.solutions import QNetwork, linear_schedule, epsilon_greedy_policy, Probe2


### Exercise - fill in the agent class

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 25-45 minutes on this exercise.
> ```

You should now fill in the methods for the `DQNAgent` class below. This is a class which is designed to handle taking steps in the environment (with an epsilon greedy policy), and updating the buffer.

1. `play_step` should be somewhat similar to the demo code you saw earlier, which sampled a batch of experiences to add to the buffer. It should:
    - Get actions (using `self.get_actions` rather than randomly sampling like we did in the demo code before)
    - Step our environment with these actions
    - Add the new experiences to the buffer
        - Some of these observations 
    - Set your new observation as `self.obs`, ready for the next step
2. `get_actions` should do the following:
    - Set `self.epsilon` according to the linear schedule function & the current global step counter
    - Sample actions according to the epsilon-greedy policy (i.e. using your `epsilon_greedy_policy` function), and return them

A small note on code practices here - the implementation below was designed to follow **separation of concerns** (SoC), a design principle used in software engineering. The `DQNAgent` class only responsible for interacting with the environment; it doesn't do anything like create the Q-network or buffer on initialization. This is further reflected in the fact that we don't pass in `args` to our DQN agent, but instead pass in all the relevant variables separately (if we were forced to pass in `args`, this would be a sign that the DQN agent class might be doing too much work!).

In [ ]:
class DQNAgent:
    """Base Agent class handling the interaction with the environment."""

    def __init__(
        self,
        envs: gym.vector.SyncVectorEnv,
        buffer: ReplayBuffer,
        q_network: QNetwork,
        start_e: float,
        end_e: float,
        exploration_fraction: float,
        total_timesteps: int,
        rng: np.random.Generator,
    ):
        self.envs = envs
        self.buffer = buffer
        self.q_network = q_network
        self.start_e = start_e
        self.end_e = end_e
        self.exploration_fraction = exploration_fraction
        self.total_timesteps = total_timesteps
        self.rng = rng

        self.step = 0  # Tracking number of steps taken (across all environments)
        self.obs, _ = self.envs.reset()  # Need a starting observation
        self.epsilon = start_e  # Starting value (will be updated in `get_actions`)

    def play_step(self) -> dict:
        """
        Carries out a single interaction step between agent & environment, and adds results to the
        replay buffer.

        Returns `infos` (list of dictionaries containing info we will log).
        """
        raise NotImplementedError()

        self.step += self.envs.num_envs
        return infos

    def get_actions(self, obs: np.ndarray) -> np.ndarray:
        """
        Samples actions according to the epsilon-greedy policy using the linear schedule for epsilon.
        """
        raise NotImplementedError()


tests.test_agent(DQNAgent)

<details><summary>Solution</summary>

```python
class DQNAgent:
    """Base Agent class handling the interaction with the environment."""

    def __init__(
        self,
        envs: gym.vector.SyncVectorEnv,
        buffer: ReplayBuffer,
        q_network: QNetwork,
        start_e: float,
        end_e: float,
        exploration_fraction: float,
        total_timesteps: int,
        rng: np.random.Generator,
    ):
        self.envs = envs
        self.buffer = buffer
        self.q_network = q_network
        self.start_e = start_e
        self.end_e = end_e
        self.exploration_fraction = exploration_fraction
        self.total_timesteps = total_timesteps
        self.rng = rng

        self.step = 0  # Tracking number of steps taken (across all environments)
        self.obs, _ = self.envs.reset()  # Need a starting observation
        self.epsilon = start_e  # Starting value (will be updated in `get_actions`)

    def play_step(self) -> dict:
        """
        Carries out a single interaction step between agent & environment, and adds results to the
        replay buffer.

        Returns `infos` (list of dictionaries containing info we will log).
        """
        self.obs = np.array(self.obs, dtype=np.float32)
        actions = self.get_actions(self.obs)
        next_obs, rewards, terminated, truncated, infos = self.envs.step(actions)

        # Get `real_next_obs` by finding all environments where we terminated & replacing `next_obs`
        # with the actual terminal states
        true_next_obs = next_obs.copy()
        for n in range(self.envs.num_envs):
            if (terminated | truncated)[n]:
                true_next_obs[n] = infos["final_observation"][n]

        self.buffer.add(self.obs, actions, rewards, terminated, true_next_obs)
        self.obs = next_obs

        self.step += self.envs.num_envs
        return infos

    def get_actions(self, obs: np.ndarray) -> np.ndarray:
        """
        Samples actions according to the epsilon-greedy policy using the linear schedule for epsilon.
        """
        self.epsilon = linear_schedule(
            self.step, self.start_e, self.end_e, self.exploration_fraction, self.total_timesteps
        )
        actions = epsilon_greedy_policy(self.envs, self.q_network, self.rng, obs, self.epsilon)
        assert actions.shape == (len(self.envs.envs),)
        return actions
```
</details>

Before we move on to the big exercise of today (completing the `DQNTrainer` class), we'll briefly discuss logging to Weights and Biases in RL, plus some general advice on what kinds of variables you should be logging.

### Logging to `wandb` in RL

In previous exercises in this chapter, we've just trained the agent, and then plotted the reward per episode after training. For small toy examples that train in a few seconds this is fine, but for longer runs we'd like to watch the run live and make sure the agent is doing something interesting (especially if we were planning to run the model overnight). Luckily, **Weights and Biases** has got us covered! When you run your experiments, you'll be able to view not only *live plots* of the loss and average reward per episode while the agent is training - you can also log and view animations, which visualise your agent's progress in real time! The code below will handle all logging.

Sadly, effective logging & debugging in RL isn't just about watching videos, since in the vast majority of cases where your algorithm has a bug, the agent will just fail to learn anything useful and the videos won't be informative. Debugging RL requires knowing what variables to log and how to interpret the results you're getting, which requires some understanding of the underlying theory! This is part of the reason why we've spent so much time discussing the theory behind DQN and other RL algorithms, rather than just giving you a black box to train.

As an example of how logged variables can be misleading and hard to interpret, consider our TD loss function in DQN. This loss function just reflects how close together the Q-network's estimates are to the experiences currently sampled from the replay buffer, which might not adequately represent what the world actually looks like. This means that once the agent starts to learn something and do better at the problem, it's expected for the loss to increase. For example, maybe the Q-network initially learned some state was bad, because an agent that reached them was just flapping around randomly and died shortly after. But now it's getting evidence that the same state is good, now that the agent that reached the state has a better idea what to do next. A higher loss is thus actually a good sign that something is happening (the agent hasn't stagnated), but it's not clear if it's learning anything useful without also checking how the total reward per episode has changed. Key point - **just looking at one variable can be misleading, we need to log multiple variables and derive a picture of what's happening from taking all of them into account!**

Some useful variables to log during DQN training are:

- TD loss, i.e. the actual loss you're backpropagating through. This should start off high and decrease pretty quickly, but may not be monotonic (i.e. temporary spikes in loss aren't necessarily a bad thing)
- SPS (steps per second), i.e. the total number of agent steps divided by the total time. This helps us debug when the environment steps are a bottleneck (won't be the case in a simple environment like this one, but might matter more when we move to more complex environments)
- Q-values, i.e. the predicted Q-values from the Q-network. Can you guess how these should behave?

<details>
<summary>Question - what do you think the Q values will do when the agent moves closer to solving the cartpole environment?</summary>

Initially they should be near zero, thanks to the randomly initialized model weights. As our episode length get closer to 500 (i.e. we can essentially solve the environment), they should tend to the limit of the total possible time-discounted reward available, which is the geometric sum $1 + \gamma + \gamma^2 + \cdots$ (since we get 1 reward for every second we stand up, and as previously discussed, the way we handle `dones` in the formula above doesn't assume a truncated environment causes future rewards to be terminated). The limit of this sum is $\frac{1}{1-\gamma}$, which for our default value $\gamma = 0.99$ is approximately 100.

Note, the Q values won't increase smoothly, they'll spike up immediately after we copy over the weights from our Q-network to our target network. This is because each time we copy over weights, our gradient changes and the Q-network rapidly "catches up" to this new target network, causing the Q values to change rapidly. However, our copying over of weights will be frequent enough that these jumps will be relatively small, and so the curve should still appear smooth.

</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# Wrap tests.test_agent so a passing test fires the beacon.
try:
    _dd_orig = tests.test_agent
    def _dd_wrapped(*args, **kwargs):
        result = _dd_orig(*args, **kwargs)
        _dd_report_complete()
        return result
    tests.test_agent = _dd_wrapped
except AttributeError:
    print('[Delta Drills] no matching test function — call _dd_report_complete() manually when done.')
